# ZS601 v018 / C / 50,000 iterations

Shared source commit `a2a6f955ca469b39f2df862a97b15821fd38da31`. All outputs write directly to your Drive.

Current formal outputs are protected: use a fresh reproduction run name.


In [ ]:
from pathlib import Path
import subprocess, sys, json
ARM = 'C'
CODE_COMMIT = 'a2a6f955ca469b39f2df862a97b15821fd38da31'
NOTEBOOK_TAG = 'v018-notebooks-v1'
REPOSITORY = "https://github.com/VISjudy/ZS601_3DGS.git"
DRIVE_ROOT = Path("/content/drive/MyDrive/LCCDataset/zs601_output/zs601-loss-virtual-v018-20260924")
RUN_NAME = "reproduction-" + ARM + "-50k-v1"  # change for each NEW run
CODE = Path("/content/zs601-v018-code-" + CODE_COMMIT[:12])
LOCAL_DATA = Path("/content/zs601-v018-reproduction-data")
OUTPUT = DRIVE_ROOT / "03-experiment-results" / RUN_NAME


## 1. Mount Drive and fetch the immutable code
Choose an L4 GPU runtime where available. Complete Drive authorization yourself.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert DRIVE_ROOT.is_dir(), "Set DRIVE_ROOT to your prepared dataset folder"
assert not OUTPUT.exists(), "Output already exists; choose a new RUN_NAME"
if not CODE.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY, str(CODE)], check=True)
subprocess.run(["git", "-C", str(CODE), "fetch", "origin", "tag", NOTEBOOK_TAG], check=True)
subprocess.run(["git", "-C", str(CODE), "checkout", "--detach", CODE_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip() == CODE_COMMIT
CURRENT_NOTEBOOK = Path("/content") / (ARM + "-reproduce.ipynb")
CURRENT_NOTEBOOK.write_bytes(subprocess.check_output(["git", "-C", str(CODE), "show", NOTEBOOK_TAG + ":notebooks/" + ARM + ".ipynb"]))


## 2. Install the recorded environment
This compiles the CUDA extensions shipped in the pinned source tree. If Colab requests a runtime restart after PyTorch installation, restart and rerun configuration/mount cells.


In [ ]:
subprocess.run([sys.executable, str(CODE / "scripts/install_colab.py")], check=True)


## 3. Stage and validate inputs
Inputs are cached on Colab local disk; outputs remain on Drive. E merges only training + 200 virtual cameras. F-real requires separate real LiDAR targets and real virtual RGB.


In [ ]:
subprocess.run([sys.executable, str(CODE / "scripts/prepare_notebook_inputs.py"),
    "--drive-root", str(DRIVE_ROOT), "--local-root", str(LOCAL_DATA), "--arm", ARM], check=True)
INPUTS = LOCAL_DATA / (ARM + "-inputs.json")
COMMAND = [sys.executable, str(CODE / "scripts/run_experiment.py"), "--arm", ARM,
    "--inputs", str(INPUTS), "--output", str(OUTPUT), "--notebook", str(CURRENT_NOTEBOOK)]
subprocess.run(COMMAND, check=True)  # preflight only, no training


## 4. Train once
50,000 iterations; 10 fixed RGB/depth/normal validation views every 5,000. The launcher rejects existing outputs and saves this notebook beside the result.


In [ ]:
subprocess.run(COMMAND + ["--execute"], check=True)


## 5. Verify completion
This verifies trajectory test completion; separate OOD evaluation is not implied.


In [ ]:
done = json.loads((OUTPUT / "completed.json").read_text())
assert done["iteration"] == 50000 and done["final_test_complete"]
assert (OUTPUT / "checkpoints/iteration_50000.pth").is_file()
for step in range(0, 50001, 5000):
    folder = OUTPUT / "val_v3" / f"iteration_{step:06d}"
    for kind in ["rgb", "depth", "normal"]:
        assert len(list(folder.glob("*_" + kind + ".png"))) == 10, (step, kind)
print(done)
print("Output:", OUTPUT)
